In [4]:
import pickle

with open("data/train_processed.pkl", "rb") as f:
    train_loaded = pickle.load(f)
with open("data/val_processed.pkl", "rb") as f:
    val_processed = pickle.load(f)

In [1]:
import torch

from transformers import (
    Blip2Processor,
    Blip2Model
)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory/1e9)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

c:\Users\Shourya\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NVIDIA GeForce RTX 4050 Laptop GPU
6.438780928
cuda


In [2]:
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

model = Blip2Model.from_pretrained("Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16).to(device)

model.eval()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.04s/it]


Blip2Model(
  (vision_model): Blip2VisionModel(
    (embeddings): Blip2VisionEmbeddings(
      (patch_embedding): Conv2d(3, 1408, kernel_size=(14, 14), stride=(14, 14))
    )
    (encoder): Blip2Encoder(
      (layers): ModuleList(
        (0-38): 39 x Blip2EncoderLayer(
          (self_attn): Blip2Attention(
            (qkv): Linear(in_features=1408, out_features=4224, bias=True)
            (projection): Linear(in_features=1408, out_features=1408, bias=True)
          )
          (layer_norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True, bias=True)
          (mlp): Blip2MLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=1408, out_features=6144, bias=True)
            (fc2): Linear(in_features=6144, out_features=1408, bias=True)
          )
          (layer_norm2): LayerNorm((1408,), eps=1e-06, elementwise_affine=True, bias=True)
        )
      )
    )
    (post_layernorm): LayerNorm((1408,), eps=1e-06, elementwise_affine=True, bias=T

In [14]:
from tqdm import tqdm
import torch

In [15]:
def extract_blip_feature(image):

    inputs = processor(
        images=image,
        return_tensors="pt"
    ).to(device, torch.float16)

    with torch.no_grad():

        vision_outputs = model.vision_model(
            pixel_values=inputs["pixel_values"]
        )

        image_embeds = vision_outputs.last_hidden_state

        image_attention_mask = torch.ones(
            image_embeds.size()[:-1],
            dtype=torch.long,
            device=device
        )

        query_tokens = model.query_tokens.expand(
            image_embeds.shape[0],
            -1,
            -1
        )

        qformer_outputs = model.qformer(
            query_embeds=query_tokens,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attention_mask,
            return_dict=True
        )

    feature = qformer_outputs.last_hidden_state.squeeze(0)

    return feature.cpu()

In [16]:
feature = extract_blip_feature(
    train_loaded[0]["image"]
)

print(feature.shape)

torch.Size([32, 768])


In [17]:
train_features = []
train_questions = []
train_labels = []

for sample in tqdm(train_loaded):
    
    feature = extract_blip_feature(sample["image"])

    train_features.append(feature)

    train_questions.append(sample["question_tokens"])

    train_labels.append(sample["answer"])

100%|██████████| 1722/1722 [03:48<00:00,  7.52it/s]


In [18]:
train_features = torch.stack(train_features)

train_questions = torch.tensor(train_questions, dtype=torch.long)

train_labels = torch.tensor(train_labels, dtype=torch.long)

print(train_features.shape)
print(train_questions.shape)
print(train_labels.shape)

torch.Size([1722, 32, 768])
torch.Size([1722, 23])
torch.Size([1722])


In [19]:
val_features = []
val_questions = []
val_labels = []

for sample in tqdm(val_processed):

    feature = extract_blip_feature(
        sample["image"]
    )

    val_features.append(feature)

    val_questions.append(
        sample["question_tokens"]
    )

    val_labels.append(
        sample["answer"]
    )

100%|██████████| 431/431 [01:09<00:00,  6.24it/s]


In [21]:
val_features = torch.stack(val_features)
val_questions = torch.tensor(val_questions)
val_labels = torch.tensor(val_labels)

In [22]:
torch.save(
    {
        "features": train_features,
        "questions": train_questions,
        "labels": train_labels
    },
    "data/train_blip_features.pt"
)

torch.save(
    {
        "features": val_features,
        "questions": val_questions,
        "labels": val_labels
    },
    "data/val_blip_features.pt"
)

In [23]:
data = torch.load("data/train_blip_features.pt")

print(data["features"].shape)
print(data["questions"].shape)
print(data["labels"].shape)

torch.Size([1722, 32, 768])
torch.Size([1722, 23])
torch.Size([1722])
